# vla-hands · Medium Notebook

**Assumes:** You've completed `01_intro.ipynb` and understand the basic graft concept.

This notebook covers:

| Section | Topic |
|---------|-------|
| 2 | All 5 appendage types — joystick, D-pad, button, multi-button, touchscreen |
| 3 | All 12 environments — single-appendage and multi-appendage |
| 4 | Training all appendages in parallel (shared VLM backbone) |
| 5 | Training curves with `plot_training_curves()` |
| 6 | GIF export — `save_rollout_gif()` and `record_expert_gif()` |
| 7 | Full benchmark suite across all trained grafts |

**Runtime:** ~20–30 min on T4 GPU

## 1 · Setup

In [ ]:
!pip install -q git+https://github.com/jerod92/project-h.git@claude/vla-robotic-hands-platform-kGiza
print('✅ vla-hands installed')

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Image as IPImage

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = 'HuggingFaceTB/SmolVLM-256M-Instruct'
print(f'Loading {MODEL_ID}...')
processor = AutoProcessor.from_pretrained(MODEL_ID)
vlm = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.float32)

text_cfg = getattr(vlm.config, 'text_config', None)
hidden_dim = (
    text_cfg.hidden_size
    if text_cfg is not None and hasattr(text_cfg, 'hidden_size')
    else vlm.config.hidden_size
)
print(f'Parameters: {sum(p.numel() for p in vlm.parameters())/1e6:.0f}M  hidden_dim={hidden_dim}')

## 2 · All Environments

12 environments across 5 appendage types. Dynamic synonym prompts prevent the model from memorising fixed phrasing.

In [ ]:
from vla_hands import (
    TargetNavEnvironment, SpaceshipNavEnvironment,
    GridWorldEnvironment, MazeEnvironment,
    ButtonPressEnvironment, MCQButtonEnvironment,
    PointingEnvironment,
)
from vla_hands.environments.fruit_catcher import FruitCatcherEnvironment
from vla_hands.environments.treasure_hunt import TreasureHuntEnvironment
from vla_hands.environments.paint_canvas import PaintCanvasEnvironment
from vla_hands.environments.whack_a_mole import WhackAMoleEnvironment
from vla_hands.environments.mcq_navigator import MCQNavigatorEnvironment

envs = [
    ('Target Nav\n(joystick)',  TargetNavEnvironment(width=192, height=192)),
    ('Spaceship\n(joystick)',   SpaceshipNavEnvironment(width=192, height=192)),
    ('Grid World\n(d-pad)',     GridWorldEnvironment(grid_size=6)),
    ('Maze\n(d-pad)',           MazeEnvironment(rows=6, cols=6)),
    ('Color Press\n(button)',   ButtonPressEnvironment(width=192, height=192)),
    ('MCQ\n(multi-button)',     MCQButtonEnvironment(width=240, height=192)),
    ('Pointing\n(touchscreen)', PointingEnvironment(width=192, height=192)),
    ('Fruit Catcher\n(joy+btn)',FruitCatcherEnvironment(width=192, height=192)),
    ('Treasure Hunt\n(dpad+btn)',TreasureHuntEnvironment()),
    ('Paint Canvas\n(touchscreen)',PaintCanvasEnvironment(width=192, height=192)),
    ('Whack-a-Mole\n(touchscreen)',WhackAMoleEnvironment(width=192, height=192)),
    ('MCQ Navigator\n(dpad+mbtn)',MCQNavigatorEnvironment()),
]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, (name, env) in zip(axes.flat, envs):
    obs = env.reset(seed=42)
    ax.imshow(obs)
    ax.set_title(name, fontsize=9)
    ax.axis('off')
plt.suptitle('All 12 vla-hands Environments', fontsize=14)
plt.tight_layout()
plt.show()

## 3 · All 5 Appendages

| Appendage | Output | Loss | Use Case |
|-----------|--------|------|----------|
| `JoystickAppendage` | `(dx, dy) ∈ [-1,1]²` | Huber | Continuous navigation |
| `DPadAppendage` | 5-way softmax | Cross-entropy | Discrete grid movement |
| `ButtonAppendage` | scalar ∈ [0,1] | BCE | Binary press/no-press |
| `MultiButtonAppendage` | N independent sigmoids | Mean BCE | Multi-choice, chord keys |
| `TouchscreenAppendage` | `(x,y) ∈ [0,1]²` | Huber | Absolute tap position |

In [ ]:
from vla_hands import (
    VLAGraft, GraftConfig,
    JoystickAppendage, DPadAppendage,
    ButtonAppendage, MultiButtonAppendage,
)
from vla_hands.appendages.touchscreen import TouchscreenAppendage

cfg = GraftConfig(feature_extraction='last')

# Detect vision encoder dim for touchscreen skip connection
vision_dim = VLAGraft.detect_vision_dim(vlm)
print(f'Vision encoder dim: {vision_dim}')

grafts = {
    'joystick':  VLAGraft(vlm, JoystickAppendage(hidden_dim), cfg),
    'dpad':      VLAGraft(vlm, DPadAppendage(hidden_dim), cfg),
    'button':    VLAGraft(vlm, ButtonAppendage(hidden_dim), cfg),
    'mcq4':      VLAGraft(vlm, MultiButtonAppendage(hidden_dim, n_buttons=4,
                                                    labels=['A','B','C','D']), cfg),
    'touchscreen': VLAGraft(vlm, TouchscreenAppendage(hidden_dim, vision_dim=vision_dim), cfg),
}

for name, g in grafts.items():
    n = g.appendage.num_parameters()
    print(f'  {name:14s}  {n:,} params  ({n/1e3:.0f}K)')

## 4 · Train All Grafts

In [ ]:
from vla_hands import TrainingCurriculum, CurriculumConfig, QUICK_CURRICULUM

def quick_train(graft, env, bc_steps=250, rl_steps=50, tag=''):
    config = CurriculumConfig(
        bc_steps=bc_steps, rl_steps=rl_steps,
        device=device,
        save_dir=f'model_checkpoints/{tag or type(env).__name__}',
        freezing_stages=QUICK_CURRICULUM,
        eval_every=bc_steps // 4,
        log_every=bc_steps // 10,
        eval_episodes=5,
    )
    return TrainingCurriculum(graft, processor, env, config).run()

print('Training helpers ready.')

In [ ]:
print('=== Joystick → Target Navigation ===')
target_env = TargetNavEnvironment(width=224, height=224, max_steps=80)
joystick_metrics = quick_train(grafts['joystick'], target_env, 300, 80, 'joystick')

In [ ]:
print('=== D-pad → Grid World ===')
grid_env = GridWorldEnvironment(grid_size=6)
dpad_metrics = quick_train(grafts['dpad'], grid_env, 300, 80, 'dpad')

In [ ]:
print('=== Button → Color Press ===')
button_env = ButtonPressEnvironment()
button_metrics = quick_train(grafts['button'], button_env, 200, 0, 'button')

In [ ]:
print('=== MultiButton → MCQ ===')
mcq_env = MCQButtonEnvironment()
mcq_metrics = quick_train(grafts['mcq4'], mcq_env, 200, 0, 'mcq')

In [ ]:
print('=== Touchscreen → Pointing (with vision skip connection) ===')
pointing_env = PointingEnvironment(width=256, height=256, n_distractors=3, n_colors=5)
ts_metrics = quick_train(grafts['touchscreen'], pointing_env, 350, 0, 'touchscreen')

## 5 · Training Curves

In [ ]:
from vla_hands.utils.viz import plot_training_curves, TrainingSummary

all_metrics = {
    'Joystick':     joystick_metrics,
    'D-pad':        dpad_metrics,
    'Button':       button_metrics,
    'Multi-Button': mcq_metrics,
    'Touchscreen':  ts_metrics,
}

# Smoothed loss curves for all 5 grafts
plot_training_curves(all_metrics, title='BC Training Loss — All Appendages', smooth=10)

In [ ]:
# Pretty-printed summary for the joystick run
summary = TrainingSummary.from_metrics(joystick_metrics)
print(summary)

## 6 · GIF Export

Export rollouts as animated GIFs — no ffmpeg needed, pure PIL.

In [ ]:
from vla_hands.utils.gif import record_expert_gif, save_rollout_gif

# Expert policy GIF (no VLM needed)
record_expert_gif(
    env=TargetNavEnvironment(width=200, height=200),
    path='expert_targetnav.gif',
    n_steps=24,
    seed=7,
    fps=8,
)

IPImage(filename='expert_targetnav.gif')

In [ ]:
# Graft inference GIF — shows what the trained model actually does
save_rollout_gif(
    graft=grafts['joystick'],
    processor=processor,
    env=TargetNavEnvironment(width=200, height=200),
    path='trained_joystick.gif',
    n_steps=24,
    seed=42,
    device=device,
    fps=6,
)

IPImage(filename='trained_joystick.gif')

## 7 · Benchmark

In [ ]:
from vla_hands import BenchmarkSuite

pairs = [
    ('Joystick',     grafts['joystick'],    TargetNavEnvironment()),
    ('D-pad',        grafts['dpad'],        GridWorldEnvironment(grid_size=6)),
    ('Button',       grafts['button'],      ButtonPressEnvironment()),
    ('MultiButton',  grafts['mcq4'],        MCQButtonEnvironment()),
    ('Touchscreen',  grafts['touchscreen'], PointingEnvironment(n_distractors=3)),
]

results = {}
for name, graft, env in pairs:
    suite = BenchmarkSuite(graft, processor, [env], device=device)
    r = suite.run_benchmark(env, n_episodes=20)
    results[name] = r
    print(f'{name:14s}  success={r.success_rate:.0%}  reward={r.mean_reward:+.2f}')

In [ ]:
names   = list(results.keys())
sr_vals = [results[n].success_rate * 100 for n in names]
rw_vals = [results[n].mean_reward for n in names]
colors  = ['#4C9BE8', '#5BC46A', '#E8804C', '#C45BE8', '#E8D04C']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

bars = ax1.bar(names, sr_vals, color=colors)
ax1.set_ylabel('Success Rate (%)')
ax1.set_title('Success Rate by Appendage')
ax1.set_ylim(0, 115)
for bar, val in zip(bars, sr_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.0f}%', ha='center', va='bottom', fontsize=11)

ax2.bar(names, rw_vals, color=colors)
ax2.set_ylabel('Mean Episode Reward')
ax2.set_title('Mean Reward by Appendage')
ax2.axhline(0, color='gray', linewidth=0.5)

plt.suptitle('vla-hands Benchmark — All Appendages', fontsize=13)
plt.tight_layout()
plt.show()

## ✅ Done!

You've trained and benchmarked all 5 appendage types and exported GIFs.

**Ready for more?** → `03_advanced.ipynb`
- LoRA fine-tuning (adapt the VLM backbone, not just the head)
- CompositeGraft (joystick + button in one forward pass)
- Auto curriculum (one call trains any appendage combination)
- New environments: FruitCatcher, WhackAMole, PaintCanvas, TreasureHunt, MCQNavigator

## 8 · Export GIFs for All Trained Grafts

In [ ]:
from vla_hands.utils.gif import record_expert_gif, save_rollout_gif
from IPython.display import Image as IPImage
import os

os.makedirs('gifs', exist_ok=True)

# One expert GIF and one trained-graft GIF per appendage
gif_pairs = [
    ('joystick',     grafts['joystick'],    TargetNavEnvironment(width=200, height=200)),
    ('dpad',         grafts['dpad'],        GridWorldEnvironment(grid_size=6)),
    ('button',       grafts['button'],      ButtonPressEnvironment(width=200, height=200)),
    ('touchscreen',  grafts['touchscreen'], PointingEnvironment(width=200, height=200, n_distractors=3)),
]

for tag, graft, env in gif_pairs:
    record_expert_gif(env, path=f'gifs/expert_{tag}.gif', n_steps=16, seed=7, fps=6)
    save_rollout_gif(graft, processor, env,
                     path=f'gifs/trained_{tag}.gif',
                     n_steps=16, seed=42, device=device, fps=5)
    print(f'{tag}: expert + trained GIFs saved')

print('\nAll GIFs written to gifs/')

In [ ]:
# Display side-by-side: expert vs trained for joystick
print('Expert (upper bound):')
display(IPImage(filename='gifs/expert_joystick.gif'))
print('Trained graft:')
display(IPImage(filename='gifs/trained_joystick.gif'))

# Touchscreen pointing — shows predicted tap positions
print('Touchscreen — trained graft on pointing:')
display(IPImage(filename='gifs/trained_touchscreen.gif'))